In [ ]:
#Install AgentNova, Ollama, Cloudflared
!sudo apt-get install zstd pciutils nano
!pip install git+https://github.com/VTSTech/AgentKthx.git --force-reinstall
# Install Ollama
!sudo curl -fsSL https://ollama.com/install.sh | sh
# Install cloudflared
!curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /tmp/cloudflared
!chmod +x /tmp/cloudflared

In [24]:
# 2. Start Ollama in the background
import subprocess, os, time
# We use OLLAMA_HOST=0.0.0.0 so the tunnel can find it
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_API_KEY'] = 'ollama-local'
os.environ['OLLAMA_CONTEXT_LENGTH'] = '262144'
subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=open('ollama.log', 'w'))
time.sleep(1)

In [ ]:
#Cloudflare Tunnel Ollama
import subprocess
import re
import time

# Start cloudflared
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ Tunnel URL: {tunnel_url}")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:

#Pull Ollama Models
#!ollama pull nemotron-3-nano:4b
#!ollama pull AgentricAi/AgentricAI_TLM:latest
#!ollama pull DedeProgames/orion:2b
#!ollama pull driaforall/tiny-agent-a:1.5b
#!ollama pull deepseek-coder:1.3b
#!ollama pull tinyllama:1.1b
#!ollama pull tinydolphin:1.1b
#!ollama pull granite3.1-moe:1b
#!ollama pull llama3.2:1b
#!ollama pull nchapman/dolphin3.0-llama3:1b
!ollama pull qwen2.5-coder:0.5b-instruct-q4_k_m
!ollama pull nchapman/dolphin3.0-qwen2.5:0.5b
!ollama pull qwen:0.5b
!ollama pull qwen2:0.5b
!ollama pull qwen2.5:0.5b
!ollama pull qwen3:0.6b
!ollama pull qwen3.5:0.8b
!ollama pull granite4:350m
!ollama pull functiongemma:270m
!ollama pull gemma3:270m
#!ollama pull smollm:135m
!ollama list

In [ ]:
#Backup Ollama AI Models to Google Drive
# Mount Google Drive first
from google.colab import drive
drive.mount('/content/drive')

# Compress ~/.ollama/models and save to Drive
import os
import shutil

MODELS_DIR = os.path.expanduser("~/.ollama/models")
DRIVE_DIR = "/content/drive/MyDrive/ollama_models_backup"
ARCHIVE_NAME = "ollama_models.tar.gz"

# Create backup directory in Drive
os.makedirs(DRIVE_DIR, exist_ok=True)

# Compress the models directory
print("Compressing ~/.ollama/models ...")
!tar -czvf {DRIVE_DIR}/{ARCHIVE_NAME} -C ~/.ollama models

print(f"\n✅ Backup saved to: {DRIVE_DIR}/{ARCHIVE_NAME}")

# Show file size
!ls -lh {DRIVE_DIR}/{ARCHIVE_NAME}

In [ ]:
#Restore Ollama AI Models from Google Drive
# Mount Google Drive first (if not already mounted)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

DRIVE_DIR = "/content/drive/MyDrive/ollama_models_backup"
ARCHIVE_NAME = "ollama_models.tar.gz"
ARCHIVE_PATH = f"{DRIVE_DIR}/{ARCHIVE_NAME}"

# Check if backup exists
if not os.path.exists(ARCHIVE_PATH):
    raise FileNotFoundError(f"Backup not found: {ARCHIVE_PATH}")

# Remove existing models (if any) to avoid conflicts
print("Cleaning existing models directory...")
!rm -rf ~/.ollama/models

# Extract to ~/.ollama (creates ~/.ollama/models)
print("Extracting models from Drive...")
!tar -xzvf {ARCHIVE_PATH} -C ~/.ollama

print(f"\n✅ Models restored to: ~/.ollama/models")

# Verify restoration
!ls -la ~/.ollama/models/

In [ ]:
#Pull BitNet Model
# 1. Mount your Drive so we can save the success
from google.colab import drive
drive.mount('/content/drive')

# 2. Get the official model (skipping the broken conversion scripts)
!mkdir -p /content/BitNet/models/BitNet-b1.58-2B-4T
!wget -O /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf \
    "https://huggingface.co/microsoft/bitnet-b1.58-2B-4T-gguf/resolve/main/ggml-model-i2_s.gguf"
!ls -lh /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf
# 3. Save it to your backup so you never have to do this again
!cp /content/BitNet/models/BitNet-b1.58-2B-4T/bitnet_2b_i2_s.gguf /content/drive/MyDrive/BitNet_Backup/

In [16]:
#BitNet llama in background
import subprocess, threading, time, os
!pkill llama-server
# 1. Prepare the Binary
!chmod +x /content/BitNet/build/bin/llama-server

# 2. Define the Launch Command
# We use the full path to ensure no "file not found" errors
#model_path = "/content/BitNet/models/Falcon3-1B-Instruct-1.58bit/ggml-model-i2_s.gguf"
model_path = "/content/BitNet/models/BitNet-b1.58-2B-4T/bitnet-b1.58-2B-4T.gguf"
bin_path = "/content/BitNet/build/bin/llama-server"

cmd = [
    bin_path,
    "-m", model_path,
    "--host", "0.0.0.0",
    "--port", "8765",
    "--ctx-size", "2048",
    "--no-mmap"
]

# 3. Launch llama-server in the background
# We redirect output to a log file so it doesn't spam the cell
with open('llama_server.log', 'w') as log_file:
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print("⏳ llama-server is warming up in the background...")
time.sleep(1) # Give the model a moment to load into RAM

⏳ llama-server is warming up in the background...


In [ ]:
#Cloudflare Tunnel Ollama (BitNet)
import subprocess
import re
import time

# Start cloudflared
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8765'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ Tunnel URL: {tunnel_url}")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:
# --- COMPILE BITNET (THE FINAL VERSION) ---
import os, shutil

# 1. Clean & Setup
%cd /content
!sudo apt-get update -y && sudo apt-get install -y clang cmake
!rm -rf /content/BitNet
!git clone --recursive https://github.com/microsoft/BitNet.git
%cd /content/BitNet

# 2. SURGERY: Fix the code bug
!sed -i '811s/int8_t \* y_col/const int8_t \* y_col/' /content/BitNet/src/ggml-bitnet-mad.cpp

# 3. CODEGEN: Create the missing header file
# We'll use the 2B model parameters since that's what you're running
print("Running BitNet CodeGen...")
!python3 utils/codegen_tl2.py --model bitnet_b1_58-3B --BM 160,320,320 --BK 96,96,96 --bm 32,32,32
# Note: codegen creates the header in the root or include/
!mkdir -p include
!cp bitnet-lut-kernels.h include/ 2>/dev/null || cp utils/bitnet-lut-kernels.h include/ 2>/dev/null || echo "Header already in place"

# 4. STATIC BUILD CONFIG
os.environ["CC"] = "clang"
os.environ["CXX"] = "clang++"

!mkdir -p build
%cd build
!cmake .. \
    -DBUILD_SHARED_LIBS=OFF \
    -DLLAMA_BUILD_SERVER=ON \
    -DCMAKE_C_FLAGS="-I/content/BitNet/include" \
    -DCMAKE_CXX_FLAGS="-I/content/BitNet/include"

# 5. EXECUTE BUILD
!make llama-server -j$(nproc)

# 6. VERIFY & BACKUP
if os.path.exists("/content/BitNet/build/bin/llama-server"):
    print("\n✅ SUCCESS! Binary is standalone.")
    !ldd /content/BitNet/build/bin/llama-server
    !cp /content/BitNet/build/bin/llama-server /content/drive/MyDrive/BitNet_Backup/llama-server-static
    print("🚀 Standalone binary saved to Google Drive.")
else:
    print("\n❌ Build failed. Checking if header exists...")
    !ls -l /content/BitNet/include/bitnet-lut-kernels.h

In [ ]:
#backup binaries
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for your BitNet assets
!mkdir -p /content/drive/MyDrive/BitNet_Backup

# 1. Save the successfully compiled binaries
!tar -cvzf /content/drive/MyDrive/BitNet_Backup/bitnet_binaries.tar.gz -C /content/BitNet/build/bin .

# 2. Save the model weights (so you don't have to download 0.4GB again)
!cp -r /content/BitNet/models /content/drive/MyDrive/BitNet_Backup/

In [ ]:
#restore binaries
from google.colab import drive
import os

# 1. Mount and Setup Folders
drive.mount('/content/drive')
!mkdir -p /content/BitNet/build/bin
!mkdir -p /content/BitNet/models

# 2. Extract Binaries and Models
!tar -xvzf /content/drive/MyDrive/BitNet_Backup/bitnet_binaries.tar.gz -C /content/BitNet/build/bin
!cp -r /content/drive/MyDrive/BitNet_Backup/models/* /content/BitNet/models/

In [ ]:
!curl http://127.0.0.1:11434
!ollama list

In [ ]:
#turboquant
import subprocess, os

# Reset shell CWD
subprocess.run(["/bin/bash", "-c", "cd /content && pwd"], check=True)

# Kill any lingering servers
subprocess.run(["/bin/bash", "-c", "kill %1 2>/dev/null; pkill -f llama-server 2>/dev/null"], stderr=subprocess.DEVNULL)

# Clean up
subprocess.run(["/bin/bash", "-c", "cd / && rm -rf /content/llama-cpp-turboquant"])

# Clone
print("Cloning feature/turboquant-kv-cache branch...")
r = subprocess.run(["git", "clone", "-b", "feature/turboquant-kv-cache",
    "https://github.com/TheTom/llama-cpp-turboquant.git",
    "/content/llama-cpp-turboquant"],
    capture_output=True, text=True, cwd="/content")
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr)

# Verify branch
r = subprocess.run(["git", "log", "--oneline", "-3"],
    capture_output=True, text=True, cwd="/content/llama-cpp-turboquant")
print("Branch:", r.stdout.strip())

In [ ]:
%%bash
cd /content/llama-cpp-turboquant

# Fix the incomplete GCC 13.3 patch (e9c54d5 missed the non-shared path)
# Still needed as of 2026-09-23 (branch a3d5603) — ggml.h:187 still has 'extern'
sed -i 's/^#    define GGML_API extern$/#    define GGML_API/' ggml/include/ggml.h

# Verify the fix
echo "=== Verify patch ==="
grep -n "define GGML_API" ggml/include/ggml.h

echo ""
echo "=== Configuring (CPU static build for Colab) ==="
cmake -B build \
  -DGGML_CUDA=OFF \
  -DLLAMA_CURL=ON \
  -DLLAMA_BUILD_SERVER=ON \
  -DBUILD_SHARED_LIBS=OFF \
  -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -10

echo ""
echo "=== Building llama-server (this takes ~5-10 min) ==="
NPROC=$(nproc)
cmake --build build --target llama-server -j${NPROC} 2>&1 | tail -15

echo ""
file build/bin/llama-server
ls -lh build/bin/llama-server
build/bin/llama-server --help 2>&1 | grep -i turbo

In [ ]:
%%bash
cd /content/llama-cpp-turboquant

# Fix the duplicate symbol: ops.cpp:22 has a definition, but ggml-turbo-quant.c:32
# already defines it. Make ops.cpp use extern instead.
sed -i '22s/GGML_API int turbo3_cpu_wht_group_size;/extern int turbo3_cpu_wht_group_size;/' ggml/src/ggml-cpu/ops.cpp

# Verify
echo "=== ops.cpp:22 after patch ==="
sed -n '22p' ggml/src/ggml-cpu/ops.cpp

# Rebuild just ops.cpp + re-link (much faster than full rebuild — ~2 min)
echo ""
echo "=== Rebuilding (incremental — ops.cpp + link only) ==="
NPROC=$(nproc)
cmake --build build --target llama-server -j${NPROC} 2>&1 | tail -15

echo ""
file build/bin/llama-server
ls -lh build/bin/llama-server
build/bin/llama-server --help 2>&1 | grep -i turbo

In [ ]:
#TurboQuant llama-server in background (reuses Ollama models)
import subprocess, threading, time, os, sys

!pkill llama-server 2>/dev/null

# TurboQuant uses Ollama's blob paths directly — no GGUF conversion needed.
# Pick a model you already pulled in the Ollama cell (cell 3).
MODEL_NAME = "qwen2.5:0.5b"  # change to any model from `ollama list`
PORT = 8764
CTX = 8192

# Resolve the Ollama blob path for this model
sys.path.insert(0, "/usr/lib/python3/dist-packages")
try:
    from agentkthx.backends.ollama_registry import find_model
    m = find_model(MODEL_NAME)
    if m and m.blob_path and os.path.exists(m.blob_path):
        model_path = str(m.blob_path)
        print(f"✅ Found Ollama blob: {model_path}")
    else:
        raise FileNotFoundError(f"Model {MODEL_NAME} not found in Ollama registry")
except Exception as e:
    print(f"❌ Could not resolve blob path via agentkthx: {e}")
    print("Available models:")
    !ollama list
    raise

bin_path = "/content/llama-cpp-turboquant/build/bin/llama-server"
cmd = [
    bin_path,
    "-m", model_path,
    "-c", str(CTX),
    "--port", str(PORT),
    "--host", "0.0.0.0",
    # TurboQuant KV cache types:
    #   turbo3 = aggressive compression (good for q4_K_M / q4_0 weights)
    #   q8_0  = safe (use for K cache with asymmetric quants)
    #   turbo4 = good for V cache with q8_0 K
    "-ctk", "turbo3",
    "-ctv", "turbo3",
    "-t", str(os.cpu_count() or 4),
]

# Launch llama-server in the background
with open('turbo_server.log', 'w') as log_file:
    server_proc = subprocess.Popen(cmd, stdout=log_file, stderr=subprocess.STDOUT)

print(f"⏳ TurboQuant llama-server starting on port {PORT}...")
time.sleep(2)  # give it a moment to load

# Quick health check
import urllib.request
try:
    resp = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
    print(f"✅ Server ready: HTTP {resp.status}")
except Exception as e:
    print(f"⏳ Still warming up (or check turbo_server.log): {e}")
    !tail -20 turbo_server.log 2>/dev/null

In [ ]:
#Cloudflare Tunnel TurboQuant
import subprocess
import re
import time

# Start cloudflared (already installed in cell 0)
proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8764'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait and capture the tunnel URL
tunnel_url = None
for _ in range(30):  # 30 second timeout
    line = proc.stdout.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    time.sleep(1)

if tunnel_url:
    print(f"✅ TurboQuant Tunnel URL: {tunnel_url}")
    print(f"   Use with: agentkthx chat --backend llama-server --model {MODEL_NAME}")
    print(f"   (set LLAMA_SERVER_BASE_URL={tunnel_url})")
else:
    print("❌ Failed to get tunnel URL")

In [ ]:
#backup TurboQuant binary
from google.colab import drive
drive.mount('/content/drive')

# Create a folder for your TurboQuant assets
!mkdir -p /content/drive/MyDrive/TurboQuant_Backup

# Save the compiled llama-server binary (no models needed — reuses Ollama blobs)
!cp /content/llama-cpp-turboquant/build/bin/llama-server /content/drive/MyDrive/TurboQuant_Backup/llama-server-turboquant
print("✅ Binary saved to Google Drive.")
!ls -lh /content/drive/MyDrive/TurboQuant_Backup/

In [ ]:
#restore TurboQuant binary
from google.colab import drive
import os

drive.mount('/content/drive')

# Restore the binary (skip the ~10 min compile on future runs)
!mkdir -p /content/llama-cpp-turboquant/build/bin
!cp /content/drive/MyDrive/TurboQuant_Backup/llama-server-turboquant /content/llama-cpp-turboquant/build/bin/llama-server
!chmod +x /content/llama-cpp-turboquant/build/bin/llama-server

# Verify
if os.path.exists("/content/llama-cpp-turboquant/build/bin/llama-server"):
    print("✅ TurboQuant binary restored.")
    !/content/llama-cpp-turboquant/build/bin/llama-server --version
else:
    print("❌ Restore failed — run the compile cell first.")